# KEN Model - Balance Sheet & Transaction flow matrix

## Run Model
Run a single baseline simulation to generate results for the balance sheet analysis.

In [1]:
%reload_ext autoreload
%autoreload 2
from model.experiment_runner import ExperimentRunner
from model.balance_sheet import create_balance_sheet, create_transition_matrix, to_latex_symbols


In [2]:
exp = ExperimentRunner()
exp_reports = exp.run(
    scenarios=["Transformation"], 
    iterations=1,
    parameter_overrides={'Oil_exports_random_fluctuations': False}
)
report = exp_reports["Transformation"][0]

>> Running simulation 1/1 (Transformation, seed=777851)                   v.Y_growth_rate_forecast[t] 0.19050854100449066 time  1
v.Y_growth_rate_forecast[t] 0.26816194169624863 time  2
v.Y_growth_rate_forecast[t] 0.06914687034605468 time  3
v.Y_growth_rate_forecast[t] 0.06756996329896259 time  4
v.Y_growth_rate_forecast[t] 0.06500015958949999 time  5
v.Y_growth_rate_forecast[t] 0.06500543068738675 time  6
v.Y_growth_rate_forecast[t] 0.07065056191177266 time  7
v.Y_growth_rate_forecast[t] 0.06780090606375531 time  8
v.Y_growth_rate_forecast[t] 0.06427271706744203 time  9
v.Y_growth_rate_forecast[t] 0.0667056195432252 time  10
v.Y_growth_rate_forecast[t] 0.06736369673500392 time  11
v.Y_growth_rate_forecast[t] 0.06234019999939762 time  12
v.Y_growth_rate_forecast[t] 0.05808138478349846 time  13
v.Y_growth_rate_forecast[t] 0.052196648711212844 time  14
v.Y_growth_rate_forecast[t] 0.0494220675691948 time  15
v.Y_growth_rate_forecast[t] 0.046967994640726166 time  16
v.Y_growth_rate_forecas

## Balance Sheet (Stock Matrix)
Shows assets and liabilities at a point in time. Financial row sums (Deposits, Loans, Bonds) should be zero. Capital is a real asset.

In [3]:
t = 40
bs, bs_sym = create_balance_sheet(report, t=t)

Balance Sheet (Symbols):


,Households,Firms,Banks,Government/CB,ROW,Sum
Capital (K),,+K,,,,+K
HH Portfolio Claims (V),+V,,-V,,,0
HH Deposits (D_HH),+D_HH,,-D_HH,,,0
Loans (L),,-L,+L,,,0
Dom Gov Bonds,,,+Bond_dom,-Bond_dom,,0
Ext Gov Bonds,,,,-Bond_ext,+Bond_ext,0
Gov Ext Assets,,,,+Gov_ext_assets,-Gov_ext_assets,0
FDI Net Stock,,-FDI_stock,,,+FDI_stock,0
"Private Firm Equity — HH (E_f,HH)",+E_firm_HH,-E_firm_HH,,,,0
"Private Firm Equity — Gov (E_f,Gov)",,-E_firm_gov,,+E_firm_gov,,0



Balance Sheet (Values in billion SAR, t=40):


,Households,Firms,Banks,Government/CB,ROW,Sum
Capital (K),0.00,84427.02,0.00,0.00,0.00,84427.02
HH Portfolio Claims (V),7995.47,0.00,-7995.47,0.00,0.00,0.00
HH Deposits (D_HH),34908.45,0.00,-34908.45,0.00,0.00,0.00
Loans (L),0.00,-34908.45,34908.45,0.00,0.00,0.00
Dom Gov Bonds,0.00,0.00,3907.04,-3907.04,0.00,0.00
Ext Gov Bonds,0.00,0.00,0.00,-2605.52,2605.52,0.00
Gov Ext Assets,0.00,0.00,0.00,4006.18,-4006.18,0.00
FDI Net Stock,0.00,-17881.48,0.00,0.00,17881.48,0.00
"Private Firm Equity — HH (E_f,HH)",25642.91,-25642.91,0.00,0.00,0.00,0.00
"Private Firm Equity — Gov (E_f,Gov)",0.00,-5994.17,0.00,5994.17,0.00,0.00


In [4]:
def _fit_to_textwidth(latex_str: str) -> str:
    """Wrap the tabular environment in \\resizebox{\\textwidth}{!}{} so the table
    always fits within the A4 portrait text width when included in a LaTeX document.
    Requires the 'graphicx' package (\\usepackage{graphicx}) in the document preamble.
    """
    return latex_str.replace(
        r'\begin{tabular}',
        r'\resizebox{\textwidth}{!}{%' + '\n' + r'\begin{tabular}'
    ).replace(
        r'\end{tabular}',
        r'\end{tabular}' + '\n' + r'}%'
    )

# Export numeric balance sheet (scaled to billions)
bs_latex = (bs / 1e6).round(1).to_latex(
    caption=f"Balance Sheet at $t={t}$ (Billion SAR)",
    label="tab:balance_sheet",
    escape=False,
    column_format='l' + 'r' * len(bs.columns),
    float_format='%.1f'
)

with open('Tables/balance_sheet.tex', 'w') as f:
    f.write(_fit_to_textwidth(bs_latex))

# Export symbolic balance sheet with LaTeX-formatted symbols
bs_sym_latex_df = to_latex_symbols(bs_sym)
bs_sym_latex = bs_sym_latex_df.to_latex(
    caption=f"Balance Sheet Symbols at $t={t}$",
    label="tab:balance_sheet_symbols",
    escape=False,
    column_format='p{5.5cm}' + 'c' * len(bs_sym_latex_df.columns)
)

with open('Tables/balance_sheet_symbols.tex', 'w') as f:
    f.write(_fit_to_textwidth(bs_sym_latex))

print("Balance sheet exported to Tables/balance_sheet.tex and Tables/balance_sheet_symbols.tex")


Balance sheet exported to Tables/balance_sheet.tex and Tables/balance_sheet_symbols.tex


## Transaction flow matrix (Flow of Funds)
Shows all income, expenditure, and balance-sheet-change flows between agents. All 85 firm sectors are aggregated into a single "Firms" column. Each row should sum to zero.

In [5]:
t = 40
tm, tm_sym = create_transition_matrix(report, t=t)

Transaction flow matrix (Symbols):


,Households,Firms (Current),Firms (Capital),Banks,Government,ROW,Sum
Consumption,-C,+C,,,,,
VAT,-VAT,,,,+VAT,,
Gov Consumption,,+GY,,,-GY,,
Gov Subsidies,,+Sub,,,-Sub,,
Priv. Investment (dom. financed),,+I_private_dom,-I_private_dom,,,,
Gov Investment,,+GI,,,-GI,,
Exports,,+EX,,,,-EX,
Imports,,-IM,,,,+IM,
Wages,+W,-W,,,,,
Distributed Profits,+distributed_profits,-distributed_profits,,,,,



Transaction flow matrix (Values in billion SAR, t=40):


,Households,Firms (Current),Firms (Capital),Banks,Government,ROW,Sum
Consumption,-5671.46,5671.46,0.00,0.00,0.00,0.00,0.00
VAT,-460.76,0.00,0.00,0.00,460.76,0.00,0.00
Gov Consumption,0.00,3085.68,0.00,0.00,-3085.68,0.00,0.00
Gov Subsidies,0.00,0.05,0.00,0.00,-0.05,0.00,0.00
Priv. Investment (dom. financed),0.00,4456.80,-4456.80,0.00,0.00,0.00,0.00
Gov Investment,0.00,244.02,0.00,0.00,-244.02,0.00,0.00
Exports,0.00,4411.13,0.00,0.00,0.00,-4411.13,0.00
Imports,0.00,-5066.72,0.00,0.00,0.00,5066.72,0.00
Wages,6691.72,-6691.72,0.00,0.00,0.00,0.00,0.00
Distributed Profits,1419.79,-1419.79,0.00,0.00,0.00,0.00,0.00


In [6]:

# Export Transaction flow matrix as LaTeX

# Export numeric transaction flow matrix (scaled to billions)
tm_latex = (tm / 1e6).round(1).to_latex(
    caption=f"Transaction Flow Matrix at $t={t}$ (Billion SAR)",
    label="tab:transaction_flow_matrix",
    escape=False,
    column_format='l' + 'r' * len(tm.columns),
    float_format='%.1f'
)

with open('Tables/transaction_flow_matrix.tex', 'w') as f:
    f.write(_fit_to_textwidth(tm_latex))

# Export symbolic transaction flow matrix with LaTeX-formatted symbols
tm_sym_latex_df = to_latex_symbols(tm_sym)
tm_sym_latex = tm_sym_latex_df.to_latex(
    caption=f"Transaction Flow Matrix Symbols at $t={t}$",
    label="tab:transaction_flow_matrix_symbols",
    escape=False,
    column_format='l' + 'c' * len(tm_sym_latex_df.columns)
)

with open('Tables/transaction_flow_matrix_symbols.tex', 'w') as f:
    f.write(_fit_to_textwidth(tm_sym_latex))

print("Transaction flow matrix exported to Tables/transaction_flow_matrix.tex and Tables/transaction_flow_matrix_symbols.tex")


Transaction flow matrix exported to Tables/transaction_flow_matrix.tex and Tables/transaction_flow_matrix_symbols.tex


## Balance Sheet Over Time
Check balance sheet consistency across multiple time periods.

In [7]:
macro = report["results_macro"]
for t_check in [0, 9, 19, -1]:
    year = macro.index[t_check] + 2020
    bs_t, _ = create_balance_sheet(report, t=t_check, display=False)
    dep_sum  = bs_t.loc['HH Portfolio Claims (V)', 'Sum']
    loan_sum = bs_t.loc['Loans (L)', 'Sum']
    bdom_sum = bs_t.loc['Dom Gov Bonds', 'Sum']
    bext_sum = bs_t.loc['Ext Gov Bonds', 'Sum']
    nfa_sum  = bs_t.loc['Gov Ext Assets', 'Sum']
    fdi_sum  = bs_t.loc['FDI Net Stock', 'Sum']
    ef_HH_sum = bs_t.loc['Private Firm Equity — HH (E_f,HH)', 'Sum']
    ef_gov_sum = bs_t.loc['Private Firm Equity — Gov (E_f,Gov)', 'Sum']
    ef_sum   = ef_HH_sum + ef_gov_sum
    eb_sum   = bs_t.loc['Bank Equity (E_b)', 'Sum']
    ofa_sum  = bs_t.loc['Other Financial Assets (OFA) Net Stock', 'Sum']
    print(f"Year {year}: K={bs_t.loc['Capital (K)', 'Firms']:>12,.0f}  "
          f"V={bs_t.loc['HH Portfolio Claims (V)', 'Households']:>12,.0f}  "
          f"Gov_ext_assets={bs_t.loc['Gov Ext Assets', 'Government/CB']:>12,.0f}  "
          f"| Row sums (all should=0): V={dep_sum:.1f} Loan={loan_sum:.1f} "
          f"BDom={bdom_sum:.1f} BExt={bext_sum:.1f} GEA={nfa_sum:.1f} FDI={fdi_sum:.1f} "
          f"E_f={ef_sum:.1f} E_b={eb_sum:.1f} OFA={ofa_sum:.1f}")

Year 2021: K=15,658,030,486  V= 444,996,160  Gov_ext_assets=3,119,358,622  | Row sums (all should=0): V=0.0 Loan=0.0 BDom=0.0 BExt=0.0 GEA=0.0 FDI=0.0 E_f=0.0 E_b=0.0 OFA=0.0
Year 2030: K=22,359,313,586  V=3,665,849,792  Gov_ext_assets=5,962,378,220  | Row sums (all should=0): V=0.0 Loan=0.0 BDom=0.0 BExt=0.0 GEA=0.0 FDI=0.0 E_f=0.0 E_b=-0.0 OFA=0.0
Year 2040: K=34,101,787,785  V=4,913,242,464  Gov_ext_assets=8,327,924,035  | Row sums (all should=0): V=0.0 Loan=0.0 BDom=0.0 BExt=0.0 GEA=0.0 FDI=0.0 E_f=0.0 E_b=-0.0 OFA=-0.0
Year 2061: K=84,427,019,078  V=7,995,470,147  Gov_ext_assets=4,006,180,220  | Row sums (all should=0): V=0.0 Loan=0.0 BDom=0.0 BExt=0.0 GEA=0.0 FDI=0.0 E_f=0.0 E_b=0.0 OFA=0.0
